# Part 2: Deep Learning Approach

This notebook represents the second part of the challenge, where we use Deep Learning techniques like Convolutional Neural Networks to classify the white blood cell images.

Please ensure your environment has the following library versions installed:

| Library      | Version      |
|--------------|--------------|
| PyTorch      | 2.2.0+cu118  |
| NumPy        | 1.26.4       |
| Pandas       | 3.0.1        |
| Seaborn      | 0.13.2       |

### Data directory structure
<pre>
data/
├── raw/
│   ├── features_extraites_train.csv
│   ├── features_v2_test.csv
│   └── features_v2_train.csv
├── test/
└── train/
</pre>

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.models import EfficientNet_B3_Weights, ResNet50_Weights, ConvNeXt_Tiny_Weights

from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import seaborn as sns
from PIL import Image
import warnings
warnings.filterwarnings('ignore')


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Working_directory = Path("data/raw")  
TRAIN_DIR = Working_directory / "train"
TEST_DIR  = Working_directory / "test"
TRAIN_CSV = Working_directory / "train_metadata.csv"
TEST_CSV  = Working_directory / "test_metadata.csv"
SAMPLE_SUB = Working_directory / "sample_submission.csv"

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

print(f"Train : {len(train_df)} images | Test : {len(test_df)} images")
print(f"Classes : {train_df['label'].nunique()}")
print()

# Encode label en entier
classes = sorted(train_df['label'].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {i: c for c, i in class_to_idx.items()}
NUM_CLASSES = len(classes)
print(f"\nNombre de classes : {NUM_CLASSES}")
print(class_to_idx)

# Data Augmentation

In this section, we define transformations such as random rotations, flips, and color jittering to artificially expand our training dataset and prevent model overfitting.

In [ ]:
IMG_SIZE = 384
BATCH_SIZE  = 32
NUM_WORKERS = 8
SAMPLER_POWER = 0.5

# data augmentation avec bruit gaussien
class AddGaussianNoise:
    def __init__(self, p=0.25, std_range=(0.003, 0.02)):
        self.p         = p
        self.std_range = std_range

    def __call__(self, tensor):
        if torch.rand(1).item() < self.p:
            std   = torch.empty(1).uniform_(*self.std_range).item()
            noise = torch.randn_like(tensor) * std
            return (tensor + noise).clamp(0, 1)
        return tensor

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE + 40, IMG_SIZE + 40)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=180),
    transforms.RandomAffine(degrees=0, translate=(0.10, 0.10), scale=(0.85, 1.15), shear=8),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.0),
    transforms.ToTensor(),
    AddGaussianNoise(p=0.25, std_range=(0.003, 0.02)),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.12), ratio=(0.3, 3.3), value="random"),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

tta_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE + 40, IMG_SIZE + 40)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=180),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class WBCDataset(Dataset):
    def __init__(self, df, img_dir, class_to_idx, transform=None, is_test=False):
        self.df           = df.reset_index(drop=True)
        self.img_dir      = Path(img_dir)
        self.class_to_idx = class_to_idx
        self.transform    = transform
        self.is_test      = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = self.img_dir / f"{row['ID']}.jpg"
        if not img_path.exists():
            img_path = self.img_dir / f"{row['ID']}"
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        if self.is_test:
            return image, row['ID']
        label = self.class_to_idx[row['label']]
        return image, label

# Test loader
test_dataset = WBCDataset(test_df, TEST_DIR, class_to_idx, val_transforms, is_test=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Test : {len(test_dataset)} images")

# CNN Model + Classification Head

Here we build our Deep Learning architecture by loading a pre-trained ConvNeXt-Tiny model and modifying its final classification layers to match our specific number of classes.

In [ ]:
from torchvision.models import ConvNeXt_Tiny_Weights

def build_model(num_classes, freeze_backbone=False):
    """ConvNeXt-Tiny pré-entraîné ImageNet."""
    model = models.convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.GELU(),
        nn.Dropout(p=0.5),
        nn.Linear(512, num_classes)
    )
    return model

model = build_model(NUM_CLASSES)
total_params = sum(p.numel() for p in model.parameters())
print(f"Paramètres totaux : {total_params:,}")
del model

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.gamma = float(gamma)
        self.ce = nn.CrossEntropyLoss(
            weight=weight,
            label_smoothing=float(label_smoothing),
            reduction="none",
        )

    def forward(self, logits, targets):
        ce = self.ce(logits, targets)
        pt = torch.exp(-ce)
        loss = ((1.0 - pt) ** self.gamma) * ce
        return loss.mean()

# Class weights 
all_labels_train = train_df['label'].map(class_to_idx).values
class_sample_counts = np.bincount(all_labels_train, minlength=NUM_CLASSES).astype(np.float32)
class_weights = torch.tensor(1.0 / np.power(class_sample_counts, 0.5))
class_weights = class_weights / class_weights.mean()
class_weights = class_weights.to(device=device, dtype=torch.float32)

criterion = FocalLoss(weight=class_weights, gamma=1.0, label_smoothing=0.05)

print("Class weights (1/sqrt(n)) :")
for i, cls in enumerate(classes):
    print(f"  {cls:10s} : {class_weights[i].item():.4f}  (n={class_sample_counts[i]:.0f})")
print(f"\nFocalLoss gamma={criterion.gamma} | weights range [{class_weights.min():.3f}, {class_weights.max():.3f}]")

# Train

We configure the training loop with mixed precision and an appropriate loss function (such as Focal Loss) to handle class imbalances while optimizing our model over multiple epochs.

In [ ]:
import gc
from torch.cuda.amp import autocast, GradScaler
from sklearn.model_selection import train_test_split

# Hyperparamètres
BASE_LR_BACKBONE   = 5e-5
BASE_LR_CLASSIFIER = 1e-3
WD                 = 0.01
FREEZE_EPOCHS      = 5
NUM_EPOCHS         = 60
MIXUP_ALPHA        = 0.4
CUTMIX_ALPHA       = 1.0
PATIENCE           = 18

print(f"Config: ConvNeXt-Base | backbone_lr={BASE_LR_BACKBONE} head_lr={BASE_LR_CLASSIFIER} "
      f"wd={WD} freeze={FREEZE_EPOCHS}ep mixup={MIXUP_ALPHA} cutmix={CUTMIX_ALPHA} "
      f"patience={PATIENCE} epochs={NUM_EPOCHS}")

# Augmentations batch-level

def mixup_data(x, y, alpha=MIXUP_ALPHA):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    index = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    return mixed_x, y, y[index], lam

def cutmix_data(x, y, alpha=CUTMIX_ALPHA):
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(x.size(0), device=x.device)
    _, _, H, W = x.shape
    cut_rat = np.sqrt(1.0 - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1, x2 = np.clip(cx - cut_w//2, 0, W), np.clip(cx + cut_w//2, 0, W)
    y1, y2 = np.clip(cy - cut_h//2, 0, H), np.clip(cy + cut_h//2, 0, H)
    mixed_x = x.clone()
    mixed_x[:, :, y1:y2, x1:x2] = x[index, :, y1:y2, x1:x2]
    lam = 1.0 - (x2 - x1) * (y2 - y1) / float(H * W)
    return mixed_x, y, y[index], lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# Fonctions train / eval

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            with autocast():
                outputs = model(images)
                loss    = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += images.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / total, correct / total, f1_score(all_labels, all_preds, average='macro', zero_division=0), all_preds, all_labels

def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        if np.random.rand() < 0.5:
            mixed_images, y_a, y_b, lam = cutmix_data(images, labels)
        else:
            mixed_images, y_a, y_b, lam = mixup_data(images, labels)
        optimizer.zero_grad()
        with autocast():
            outputs = model(mixed_images)
            loss    = mixup_criterion(criterion, outputs, y_a, y_b, lam)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += images.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / total, correct / total, f1_score(all_labels, all_preds, average='macro', zero_division=0)

# Split train/val
train_data, val_data = train_test_split(
    train_df, test_size=0.20, stratify=train_df['label'], random_state=SEED
)

train_dataset = WBCDataset(train_data, TRAIN_DIR, class_to_idx, train_transforms)
val_dataset   = WBCDataset(val_data,   TRAIN_DIR, class_to_idx, val_transforms)

# Sampler
labels_train = train_data['label'].map(class_to_idx).values
counts = np.bincount(labels_train, minlength=NUM_CLASSES).astype(np.float32)
sampler_weights = 1.0 / np.power(counts + 1e-6, SAMPLER_POWER)
sample_w = sampler_weights[labels_train]
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_w, dtype=torch.double),
    num_samples=len(sample_w), replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {len(train_data)} | Val: {len(val_data)}")

# Modèle frais
torch.cuda.empty_cache()
gc.collect()
model = build_model(NUM_CLASSES, freeze_backbone=False).to(device)
scaler = GradScaler()

backbone_params   = [p for n, p in model.named_parameters() if 'classifier' not in n]
classifier_params = [p for n, p in model.named_parameters() if 'classifier' in n]

history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}
best_val_f1 = 0.0
patience_counter = 0
SAVE_PATH = Working_directory / "best_model_convnext_base.pth"

for epoch in range(1, NUM_EPOCHS + 1):
    if epoch == 1:
        for p in backbone_params:
            p.requires_grad = False
        optimizer = optim.AdamW(
            [{'params': classifier_params, 'lr': BASE_LR_CLASSIFIER}],
            weight_decay=WD
        )
        print("Backbone FROZEN")

    if epoch == FREEZE_EPOCHS + 1:
        for p in backbone_params:
            p.requires_grad = True
        optimizer = optim.AdamW([
            {'params': backbone_params,   'lr': BASE_LR_BACKBONE},
            {'params': classifier_params, 'lr': BASE_LR_CLASSIFIER},
        ], weight_decay=WD)
        scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=15, T_mult=1, eta_min=1e-6
        )
        print("Backbone UNFROZEN")

    if epoch > FREEZE_EPOCHS:
        scheduler.step(epoch - FREEZE_EPOCHS)

    t_loss, _, t_f1 = train_one_epoch(model, train_loader, optimizer, criterion, scaler, device)
    v_loss, _, v_f1, _, _ = evaluate(model, val_loader, criterion, device)

    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)
    history['train_f1'].append(t_f1)
    history['val_f1'].append(v_f1)

    if v_f1 > best_val_f1:
        best_val_f1 = v_f1
        torch.save(model.state_dict(), SAVE_PATH)
        patience_counter = 0
        star = " ★"
    else:
        patience_counter += 1
        star = ""

    lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | LR {lr:.2e} | "
          f"Train Loss {t_loss:.4f} F1 {t_f1:.4f} | "
          f"Val Loss {v_loss:.4f} F1 {v_f1:.4f}{star}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

print(f"\nBest val F1: {best_val_f1:.4f}")

# t-SNE Projection

This section visualizes the high-dimensional features extracted by the trained CNN using t-SNE (or UMAP) to demonstrate how well the model separates the different white blood cell classes in feature space.

In [ ]:
from sklearn.manifold import TSNE
try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False

def get_embeddings(model, loader, device, max_samples=3000):
    model.eval()
    embeddings, labels = [], []
    total = 0

    activation = {}
    def hook_fn(module, input, output):
        activation['embedding'] = output.flatten(1).detach().cpu()

    hook = model.avgpool.register_forward_hook(hook_fn)

    with torch.no_grad():
        for images, lbls in loader:
            if total >= max_samples:
                break
            images = images.to(device, non_blocking=True)
            with autocast():
                _ = model(images)
            embeddings.append(activation['embedding'].numpy())
            labels.extend(lbls.numpy())
            total += len(lbls)

    hook.remove()
    return np.vstack(embeddings)[:max_samples], np.array(labels)[:max_samples]

# Charger le meilleur modèle
model = build_model(NUM_CLASSES).to(device)
model.load_state_dict(torch.load(SAVE_PATH))

print("Extraction des embeddings sur le val set...")
embeddings, labels_emb = get_embeddings(model, val_loader, device, max_samples=3000)
print(f"Shape embeddings : {embeddings.shape}")

print("\nt-SNE en cours...")
tsne = TSNE(n_components=2, perplexity=40, random_state=SEED, n_jobs=-1)
emb_2d_tsne = tsne.fit_transform(embeddings)

if HAS_UMAP:
    print("UMAP en cours...")
    reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.1, random_state=SEED)
    emb_2d_umap = reducer.fit_transform(embeddings)

def plot_embeddings(emb_2d, labels, classes, title, ax):
    palette = plt.cm.get_cmap('tab20', len(classes))
    for i, cls in enumerate(classes):
        mask = labels == i
        ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1],
                   label=cls, alpha=0.6, s=8, color=palette(i))
    ax.set_title(title, fontsize=13)
    ax.legend(markerscale=3, fontsize=7, loc='best',
              bbox_to_anchor=(1.01, 1), borderaxespad=0)
    ax.set_xticks([]); ax.set_yticks([])

if HAS_UMAP:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
    plot_embeddings(emb_2d_tsne, labels_emb, classes, "t-SNE — ConvNeXt-Base", ax1)
    plot_embeddings(emb_2d_umap, labels_emb, classes, "UMAP — ConvNeXt-Base", ax2)
else:
    fig, ax1 = plt.subplots(1, 1, figsize=(12, 8))
    plot_embeddings(emb_2d_tsne, labels_emb, classes, "t-SNE — ConvNeXt-Base", ax1)

plt.suptitle("Projection des embeddings (validation set)", fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig(Working_directory / "embeddings_projection.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Courbes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'],   label='Val')
ax1.set_title('Loss'); ax1.legend(); ax1.set_xlabel('Epoch')

ax2.plot(history['train_f1'], label='Train')
ax2.plot(history['val_f1'],   label='Val')
ax2.set_title('F1 Macro'); ax2.legend(); ax2.set_xlabel('Epoch')
plt.tight_layout(); plt.show()

# Évaluation finale
model.load_state_dict(torch.load(SAVE_PATH))
_, val_acc, val_f1, val_preds, val_labels = evaluate(model, val_loader, criterion, device)

print(f"\nF1 Macro Val (meilleur modèle) : {val_f1:.4f}")
print(f"Accuracy Val : {val_acc:.4f}")
print()
print(classification_report(val_labels, val_preds, target_names=classes, zero_division=0))

# Matrice de confusion
cm = confusion_matrix(val_labels, val_preds)
plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=classes, yticklabels=classes, cmap='Blues')
plt.title('Matrice de confusion (validation)')
plt.ylabel('Vrai label'); plt.xlabel('Prédit')
plt.tight_layout(); plt.show()

cm_norm = confusion_matrix(val_labels, val_preds, normalize='true')
plt.figure(figsize=(14, 12))
sns.heatmap(cm_norm, annot=True, fmt='.2f', xticklabels=classes,
            yticklabels=classes, cmap='Blues', vmin=0, vmax=1)
plt.title('Matrice de confusion normalisée (validation)')
plt.ylabel('Vrai label'); plt.xlabel('Prédit')
plt.tight_layout(); plt.show()

# TTA and Submission

Finally, we apply Test-Time Augmentation (TTA) to improve robustness during inference on the test set and save the generated predictions into a final submission file.

In [ ]:
TTA_ROUNDS = 8
T = 0.8

model = build_model(NUM_CLASSES).to(device)
model.load_state_dict(torch.load(SAVE_PATH))
model.eval()

all_ids = []
all_probs = []

with torch.no_grad():
    for images, ids in test_loader:
        images = images.to(device, non_blocking=True)
        with autocast():
            outputs = model(images) / T
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        all_ids.extend(ids)
        all_probs.append(probs)

all_probs = np.vstack(all_probs)

tta_dataset = WBCDataset(test_df, TEST_DIR, class_to_idx, tta_transforms, is_test=True)
tta_loader  = DataLoader(tta_dataset, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

for tta_round in range(TTA_ROUNDS):
    round_probs = []
    with torch.no_grad():
        for images, _ in tta_loader:
            images = images.to(device, non_blocking=True)
            with autocast():
                outputs = model(images) / T
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            round_probs.append(probs)
    all_probs += np.vstack(round_probs)
    print(f"TTA round {tta_round + 1}/{TTA_ROUNDS} done")

all_probs /= (1 + TTA_ROUNDS)
all_preds = [idx_to_class[p] for p in np.argmax(all_probs, axis=1)]

submission = pd.DataFrame({'ID': all_ids, 'label': all_preds})
sample_sub = pd.read_csv(SAMPLE_SUB)
submission = sample_sub[['ID']].merge(submission, on='ID', how='left')
submission.to_csv(Working_directory / "submission/submission_convnext_base_tta.csv", index=False)
print("\nFichier de soumission généré !")
print(submission['label'].value_counts())